# 실습 1. Python 텍스트 데이터 처리 (Day 1 / 90분)

> 시나리오: **쇼핑몰 리뷰 데이터 정제** — `data/reviews.csv`
>
> Day 1(Pandas·정규식)은 이 저장소의 **로컬 노트북**으로 진행한다.

## Task
1. CSV 로드 후 **결측·이상치 리포트** 작성
2. `text` 컬럼 정제 — URL/특수문자/공백 정규화
3. **정규식**으로 광고 패턴(`[광고]`, `^협찬`, `구매처:`) 제거
4. `rating` 별 평균 텍스트 길이 / 단어 수 집계
5. 정제 결과를 `reviews_clean.parquet` 으로 저장

In [11]:
import pandas as pd

# data/reviews.csv 가 없으면: 터미널에서 `python data/make_reviews.py`
df = pd.read_csv("../data/reviews.csv")
print(df.shape)
df.head()

(10000, 5)


,id,user,text,rating,created_at
0,1,user087,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,2.0,2026/06/18 02:37
1,2,user072,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,4.0,23.02.2026
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14T10:17:00
3,4,user045,[광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡,4.0,NaN
4,5,user047,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,5.0,2026-05-07


## 1) 결측·이상치 리포트

In [ ]:
df.info()
print(\
print(\

# created_at 은 형식이 제각각 → errors='coerce' 로 파싱 실패는 NaT 처리
df[\
] = pd.to_datetime(df[\
], errors=\
, format=\
)
nat_count = df[\
].isna().sum()
print(\
if nat_count > 0:
    df = df.dropna(subset=[\
]).copy()
    print(f\

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          10000 non-null  int64  
 1   user        10000 non-null  str    
 2   text        10000 non-null  str    
 3   rating      9494 non-null   float64
 4   created_at  8000 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 1.3 MB

결측치:
 id               0
user             0
text             0
rating         506
created_at    2000
dtype: int64

rating 분포:
 rating
4.0    2387
5.0    2363
1.0    1689
2.0    1663
3.0    1392
NaN     506
Name: count, dtype: int64

날짜 파싱 실패(NaT): 2000 건


## 2) 텍스트 정제 파이프라인 (슬라이드 `clean()` 참고)

`.str` 로 컬럼 전체를 한 번에 정제한다. 정규식 패턴 풀이:
- `https?://\S+` → http/https 로 시작하는 URL (공백 전까지)
- `[^\w가-힣\s]` → 단어문자(`\w`)·한글(`가-힣`)·공백(`\s`) **이 아닌** 것 = 특수문자/이모지
- `\s+` → 연속 공백을 1칸으로

In [13]:
import re

def clean(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .str.strip()
         .str.replace(r"https?://\S+", "", regex=True)        # URL 제거
         .str.replace(r"[^\w가-힣\s]", " ", regex=True)        # 특수문자 제거 (가-힣 = U+AC00~U+D7A3)
         .str.replace(r"\s+", " ", regex=True)                 # 공백 정규화
         .str.strip()
    )

df["clean_text"] = clean(df["text"])
df[["text", "clean_text"]].head()

,text,clean_text
0,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,협찬 받아 작성한 후기입니다 생각보다 너무 작고 부실해요
1,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,협찬 받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요
2,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로
3,[광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡,광고 지금 구매하면 사은품 증정 포장도 꼼꼼하고 품질이 기대 이상이에요
4,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,광고 지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요


In [ ]:
AD = re.compile(r\
is_ad = df[\
].str.contains(AD, na=False)
ad_count = int(is_ad.sum())
remaining = int((~is_ad).sum())
print(f\
print(f\
if ad_count > 0:
    df = df[~is_ad].copy()
    print(f\

광고성 리뷰: 4961건
광고를 제외한 리뷰: 5039건
